# Февраль: 2 ИНН по каждой аномалии (Excel -> Озеро)

Тетрадка показывает для каждой аномалии по 2 ИНН:
1. сначала записи из Excel,
2. затем записи из Озера (ODS).

Аномалии:
- ИНН с несколькими договорами;
- ИНН с несколькими наименованиями;
- Наименование с несколькими ИНН;
- Смена договора в феврале (закрытие и открытие другого).

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Параметры
excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
month_start = '2026-02-01'
month_end = (pd.to_datetime(month_start) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

excel_inn_col = 'ИНН'
excel_contract_col = 'Номер договора'
excel_name_col = 'Наименование'

top_n_inn = 2

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
print('Impala connection initialized')

In [ ]:
def normalize_id(v):
    """
    Нормализация ИНН для UL/IP:
    - валидные длины: 10 (UL) и 12 (IP/FL)
    - если потерян только ведущий ноль: 9 -> zfill(10), 11 -> zfill(12)
    - trailing нули НЕ добавляем и НЕ удаляем (кроме .0 от Excel scientific/float)
    """
    if pd.isna(v):
        return None

    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None

    # Частый кейс Excel: scientific notation / float-like строка
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)

    # Оставляем только цифры
    s = re.sub(r"\D", '', s)
    if s == '':
        return None

    # Нормальные длины
    if len(s) in (10, 12):
        return s

    # Восстанавливаем только потерянный ведущий 0
    if len(s) == 9:
        return s.zfill(10)
    if len(s) == 11:
        return s.zfill(12)

    # Остальные длины считаем невалидными
    return None


def inn_type(inn):
    if inn is None:
        return None
    return 'UL_10' if len(inn) == 10 else ('IP_FL_12' if len(inn) == 12 else 'UNKNOWN')


def normalize_text(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', ' ')
    s = re.sub(r"\s+", ' ', s)
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    return s


def detect_excel_header_row(path, required_cols, scan_rows=30):
    raw = pd.read_excel(path, header=None)
    req = set([str(c).strip() for c in required_cols])
    for i in range(min(scan_rows, len(raw))):
        vals = set([str(x).strip() for x in raw.iloc[i].tolist()])
        if len(req & vals) >= 2:
            return i
    return 0


def to_sorted_top(vals, n=2):
    return sorted(list(vals))[:n]

## 0) Загрузка Excel и Озера

In [ ]:
header_row = detect_excel_header_row(excel_path, [excel_inn_col, excel_contract_col, excel_name_col])
excel_raw = pd.read_excel(excel_path, header=header_row)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

excel_id_col = 'ID договора' if 'ID договора' in excel_raw.columns else None

excel_df = pd.DataFrame({
    'inn': excel_raw[excel_inn_col].apply(normalize_id),
    'contract_number': excel_raw[excel_contract_col].apply(normalize_text),
    'client_name': excel_raw[excel_name_col].apply(normalize_text),
    'excel_agr_id': excel_raw[excel_id_col].apply(normalize_id) if excel_id_col else None,
})
excel_df['inn_type'] = excel_df['inn'].apply(inn_type)
excel_df = excel_df.dropna(subset=['inn']).drop_duplicates()

with imp:
    ods_all_raw = imp.fetch("""
        select
            cast(c.c_inn as string) as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where c.c_inn is not null
    """)

    ods_sa_active_raw = imp.fetch(f"""
        select
            cast(c.c_inn as string) as inn,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where a.acq_class = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and c.c_inn is not null
    """)

ods_all_df = pd.DataFrame({
    'inn': ods_all_raw['inn'].apply(normalize_id),
    'contract_number': ods_all_raw['contract_number'].apply(normalize_text),
    'client_name': ods_all_raw['client_name'].apply(normalize_text),
    'acq_class': ods_all_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_all_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_all_raw['d_valid_to'], errors='coerce').dt.date,
})
ods_all_df['inn_type'] = ods_all_df['inn'].apply(inn_type)
ods_all_df = ods_all_df.dropna(subset=['inn']).drop_duplicates()

ods_sa_active_df = pd.DataFrame({
    'inn': ods_sa_active_raw['inn'].apply(normalize_id),
    'contract_number': ods_sa_active_raw['contract_number'].apply(normalize_text),
    'client_name': ods_sa_active_raw['client_name'].apply(normalize_text),
    'acq_class': ods_sa_active_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_sa_active_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_sa_active_raw['d_valid_to'], errors='coerce').dt.date,
})
ods_sa_active_df['inn_type'] = ods_sa_active_df['inn'].apply(inn_type)
ods_sa_active_df = ods_sa_active_df.dropna(subset=['inn']).drop_duplicates()

display(pd.DataFrame([
    {'source': 'excel', 'rows': len(excel_df), 'unique_inn': excel_df['inn'].nunique(), 'ul_10': int((excel_df['inn_type'] == 'UL_10').sum()), 'ip_fl_12': int((excel_df['inn_type'] == 'IP_FL_12').sum()), 'header_row': header_row},
    {'source': 'ods_sa_active', 'rows': len(ods_sa_active_df), 'unique_inn': ods_sa_active_df['inn'].nunique(), 'ul_10': int((ods_sa_active_df['inn_type'] == 'UL_10').sum()), 'ip_fl_12': int((ods_sa_active_df['inn_type'] == 'IP_FL_12').sum()), 'header_row': None},
    {'source': 'ods_all', 'rows': len(ods_all_df), 'unique_inn': ods_all_df['inn'].nunique(), 'ul_10': int((ods_all_df['inn_type'] == 'UL_10').sum()), 'ip_fl_12': int((ods_all_df['inn_type'] == 'IP_FL_12').sum()), 'header_row': None},
]))

In [ ]:
def show_source_rows(title, df, inn_pick, cols=None):
    print(f"\n=== {title} ===")
    print('selected_inn:', inn_pick)
    out = df[df['inn'].isin(inn_pick)].copy()
    out = out.sort_values(['inn', 'contract_number', 'client_name']).drop_duplicates()
    if cols is not None:
        out = out[[c for c in cols if c in out.columns]]
    display(out)


def show_cross_rows(title, inn_pick):
    print(f"\n=== {title} ===")
    print('selected_inn:', inn_pick)
    print('\nExcel rows:')
    display(
        excel_df[excel_df['inn'].isin(inn_pick)]
        .sort_values(['inn', 'contract_number', 'client_name'])
        .drop_duplicates()
    )
    print('\nLake rows (SA active):')
    display(
        ods_sa_active_df[ods_sa_active_df['inn'].isin(inn_pick)]
        .sort_values(['inn', 'contract_number', 'client_name'])
        .drop_duplicates()
    )

## A) Excel analytics

In [ ]:
# Excel | rule1: ИНН с несколькими договорами
excel_multi_contract = (
    excel_df.assign(contract_key=excel_df['contract_number'].fillna('__NULL__'))
    .groupby('inn', as_index=False)
    .agg(contract_cnt=('contract_key', 'nunique'))
)
excel_multi_contract_inn = set(excel_multi_contract[excel_multi_contract['contract_cnt'] > 1]['inn'].tolist())
excel_pick_rule1 = to_sorted_top(excel_multi_contract_inn, n=top_n_inn)

show_source_rows(
    'Excel | rule1_inn_multi_contracts',
    excel_df,
    excel_pick_rule1,
    cols=['inn', 'contract_number', 'client_name']
)

# Excel | rule2a: ИНН с несколькими наименованиями
excel_inn_multi_name = (
    excel_df.dropna(subset=['inn', 'client_name'])
    .groupby('inn', as_index=False)
    .agg(name_cnt=('client_name', 'nunique'))
)
excel_inn_multi_name_inn = set(excel_inn_multi_name[excel_inn_multi_name['name_cnt'] > 1]['inn'].tolist())
excel_pick_rule2a = to_sorted_top(excel_inn_multi_name_inn, n=top_n_inn)

show_source_rows(
    'Excel | rule2_inn_multi_names',
    excel_df,
    excel_pick_rule2a,
    cols=['inn', 'contract_number', 'client_name']
)

# Excel | rule2b: Наименование с несколькими ИНН
excel_name_multi_inn = (
    excel_df.dropna(subset=['client_name', 'inn'])
    .groupby('client_name', as_index=False)
    .agg(inn_cnt=('inn', 'nunique'))
)
excel_bad_names = set(excel_name_multi_inn[excel_name_multi_inn['inn_cnt'] > 1]['client_name'].tolist())
excel_name_multi_inn_inn = set(excel_df[excel_df['client_name'].isin(excel_bad_names)]['inn'].dropna().tolist())
excel_pick_rule2b = to_sorted_top(excel_name_multi_inn_inn, n=top_n_inn)

show_source_rows(
    'Excel | rule2_names_multi_inn',
    excel_df,
    excel_pick_rule2b,
    cols=['inn', 'contract_number', 'client_name']
)

## B) Lake analytics (SA-active filter)

In [ ]:
# Lake | rule1: ИНН с несколькими договорами (SA-active)
ods_multi_contract = (
    ods_sa_active_df.assign(contract_key=ods_sa_active_df['contract_number'].fillna('__NULL__'))
    .groupby('inn', as_index=False)
    .agg(contract_cnt=('contract_key', 'nunique'))
)
ods_multi_contract_inn = set(ods_multi_contract[ods_multi_contract['contract_cnt'] > 1]['inn'].tolist())
ods_pick_rule1 = to_sorted_top(ods_multi_contract_inn, n=top_n_inn)

show_source_rows(
    'Lake | rule1_inn_multi_contracts',
    ods_sa_active_df,
    ods_pick_rule1,
    cols=['inn', 'contract_number', 'client_name', 'acq_class', 'd_valid_from', 'd_valid_to']
)

# Lake | rule2a: ИНН с несколькими наименованиями
ods_inn_multi_name = (
    ods_sa_active_df.dropna(subset=['inn', 'client_name'])
    .groupby('inn', as_index=False)
    .agg(name_cnt=('client_name', 'nunique'))
)
ods_inn_multi_name_inn = set(ods_inn_multi_name[ods_inn_multi_name['name_cnt'] > 1]['inn'].tolist())
ods_pick_rule2a = to_sorted_top(ods_inn_multi_name_inn, n=top_n_inn)

show_source_rows(
    'Lake | rule2_inn_multi_names',
    ods_sa_active_df,
    ods_pick_rule2a,
    cols=['inn', 'contract_number', 'client_name', 'acq_class', 'd_valid_from', 'd_valid_to']
)

# Lake | rule2b: Наименование с несколькими ИНН
ods_name_multi_inn = (
    ods_sa_active_df.dropna(subset=['client_name', 'inn'])
    .groupby('client_name', as_index=False)
    .agg(inn_cnt=('inn', 'nunique'))
)
ods_bad_names = set(ods_name_multi_inn[ods_name_multi_inn['inn_cnt'] > 1]['client_name'].tolist())
ods_name_multi_inn_inn = set(ods_sa_active_df[ods_sa_active_df['client_name'].isin(ods_bad_names)]['inn'].dropna().tolist())
ods_pick_rule2b = to_sorted_top(ods_name_multi_inn_inn, n=top_n_inn)

show_source_rows(
    'Lake | rule2_names_multi_inn',
    ods_sa_active_df,
    ods_pick_rule2b,
    cols=['inn', 'contract_number', 'client_name', 'acq_class', 'd_valid_from', 'd_valid_to']
)

## C) Cross-source check: only_excel / only_lake (2 и 2)

In [ ]:
excel_inn_set = set(excel_df['inn'].dropna().tolist())
lake_inn_set = set(ods_sa_active_df['inn'].dropna().tolist())

common_inn = excel_inn_set & lake_inn_set
only_excel_inn = sorted(list(excel_inn_set - lake_inn_set))
only_lake_inn = sorted(list(lake_inn_set - excel_inn_set))

inn_match_summary = pd.DataFrame([{
    'excel_unique_inn': len(excel_inn_set),
    'lake_unique_inn': len(lake_inn_set),
    'common_inn': len(common_inn),
    'only_excel_inn': len(only_excel_inn),
    'only_lake_inn': len(only_lake_inn),
}])

display(inn_match_summary)

pick_only_excel = only_excel_inn[:2]
pick_only_lake = only_lake_inn[:2]

show_cross_rows('Cross | only_excel (2 INN)', pick_only_excel)
show_cross_rows('Cross | only_lake (2 INN)', pick_only_lake)

## D) Дополнительная диагностика: смена договора в феврале (ODS all)

In [ ]:
month_start_dt = pd.to_datetime(month_start).date()
month_end_dt = pd.to_datetime(month_end).date()

closed_in_feb = ods_all_df[
    ods_all_df['d_valid_to'].notna() &
    (ods_all_df['d_valid_to'] >= month_start_dt) &
    (ods_all_df['d_valid_to'] <= month_end_dt)
][['inn', 'contract_number', 'd_valid_to']].drop_duplicates()

opened_in_feb = ods_all_df[
    ods_all_df['d_valid_from'].notna() &
    (ods_all_df['d_valid_from'] >= month_start_dt) &
    (ods_all_df['d_valid_from'] <= month_end_dt)
][['inn', 'contract_number', 'd_valid_from']].drop_duplicates()

switch_df = closed_in_feb.merge(
    opened_in_feb[['inn', 'contract_number', 'd_valid_from']],
    on='inn', how='inner', suffixes=('_closed', '_opened')
)
switch_df = switch_df[switch_df['contract_number_closed'] != switch_df['contract_number_opened']]
switch_inn = set(switch_df['inn'].dropna().tolist())

pick_switch = to_sorted_top(switch_inn, top_n_inn)
show_source_rows(
    'Lake | rule4_contract_switch_within_feb',
    ods_all_df,
    pick_switch,
    cols=['inn', 'contract_number', 'client_name', 'acq_class', 'd_valid_from', 'd_valid_to']
)

print('\nSwitch details for selected INN:')
display(switch_df[switch_df['inn'].isin(pick_switch)].head(100))

## E) Шаблоны ручной проверки клиента по ИНН

In [ ]:
# Шаблон 1: проверка всех строк клиента по ИНН в Excel
# Вставьте нужный ИНН и запустите ячейку
# target_inn_excel = '7708044880'

target_inn_excel = None

if target_inn_excel is None:
    print('Укажите target_inn_excel в ячейке и перезапустите.')
else:
    target_inn_excel = normalize_id(target_inn_excel)
    print('target_inn_excel:', target_inn_excel)

    print('\nExcel (normalized view):')
    display(
        excel_df[excel_df['inn'] == target_inn_excel]
        .sort_values(['inn', 'contract_number', 'client_name'])
        .drop_duplicates()
    )

    print('\nExcel raw rows (all columns):')
    display(
        excel_raw[excel_raw[excel_inn_col].apply(normalize_id) == target_inn_excel]
    )


# Шаблон 2: проверка всех строк клиента по ИНН в Озере
# Вставьте нужный ИНН и запустите ячейку
# target_inn_lake = '7708044880'

target_inn_lake = None

if target_inn_lake is None:
    print('Укажите target_inn_lake в ячейке и перезапустите.')
else:
    target_inn_lake = normalize_id(target_inn_lake)
    print('target_inn_lake:', target_inn_lake)

    print('\nLake SA-active rows:')
    display(
        ods_sa_active_df[ods_sa_active_df['inn'] == target_inn_lake]
        .sort_values(['d_valid_from', 'contract_number'])
        .drop_duplicates()
    )

    print('\nLake ALL rows (expanded diagnostics):')
    display(
        ods_all_df[ods_all_df['inn'] == target_inn_lake]
        .sort_values(['d_valid_from', 'contract_number'])
        .drop_duplicates()
    )

## F) Финальный cross-source блок: 2 only_excel ИНН и 2 only_lake ИНН

In [ ]:
excel_inn_set = set(excel_df['inn'].dropna().tolist())
lake_inn_set = set(ods_sa_active_df['inn'].dropna().tolist())

common_inn = excel_inn_set & lake_inn_set
only_excel_inn = sorted(list(excel_inn_set - lake_inn_set))
only_lake_inn = sorted(list(lake_inn_set - excel_inn_set))

cross_summary = pd.DataFrame([{
    'excel_unique_inn': len(excel_inn_set),
    'lake_unique_inn': len(lake_inn_set),
    'common_inn': len(common_inn),
    'only_excel_inn': len(only_excel_inn),
    'only_lake_inn': len(only_lake_inn),
}])
display(cross_summary)

pick_only_excel = only_excel_inn[:2]
pick_only_lake = only_lake_inn[:2]

show_cross_rows('Cross | only_excel (2 INN)', pick_only_excel)
show_cross_rows('Cross | only_lake (2 INN)', pick_only_lake)

## G) Unique agr_id check (Excel ID договора vs Озеро abs_agr_id)

In [ ]:
with imp:
    ods_ids_all_raw = imp.fetch("""
        select
            cast(c.c_inn as string) as inn,
            cast(a.abs_agr_id as string) as lake_agr_id,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where c.c_inn is not null
    """)

    ods_ids_sa_active_raw = imp.fetch(f"""
        select
            cast(c.c_inn as string) as inn,
            cast(a.abs_agr_id as string) as lake_agr_id,
            cast(a.c_agr_number as string) as contract_number,
            cast(c.c_cmp_name as string) as client_name,
            cast(a.acq_class as string) as acq_class,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from ods_alpha.scd1_agreements a
        join ods_alpha.scd1_companies c
          on c.n_cmp = a.n_cmp_client
        where a.acq_class = 'SA'
          and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
          and c.c_inn is not null
    """)

ods_ids_all_df = pd.DataFrame({
    'inn': ods_ids_all_raw['inn'].apply(normalize_id),
    'lake_agr_id': ods_ids_all_raw['lake_agr_id'].apply(normalize_id),
    'contract_number': ods_ids_all_raw['contract_number'].apply(normalize_text),
    'client_name': ods_ids_all_raw['client_name'].apply(normalize_text),
    'acq_class': ods_ids_all_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_ids_all_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_ids_all_raw['d_valid_to'], errors='coerce').dt.date,
}).dropna(subset=['inn']).drop_duplicates()

ods_ids_sa_active_df = pd.DataFrame({
    'inn': ods_ids_sa_active_raw['inn'].apply(normalize_id),
    'lake_agr_id': ods_ids_sa_active_raw['lake_agr_id'].apply(normalize_id),
    'contract_number': ods_ids_sa_active_raw['contract_number'].apply(normalize_text),
    'client_name': ods_ids_sa_active_raw['client_name'].apply(normalize_text),
    'acq_class': ods_ids_sa_active_raw['acq_class'].apply(normalize_text),
    'd_valid_from': pd.to_datetime(ods_ids_sa_active_raw['d_valid_from'], errors='coerce').dt.date,
    'd_valid_to': pd.to_datetime(ods_ids_sa_active_raw['d_valid_to'], errors='coerce').dt.date,
}).dropna(subset=['inn']).drop_duplicates()

excel_ids_df = excel_df.copy()
excel_ids_df['excel_agr_id'] = excel_ids_df['excel_agr_id'].apply(normalize_id)


def id_stats(df, id_col, source):
    tmp = df.copy()
    rows = len(tmp)
    null_id = int(tmp[id_col].isna().sum())
    nonnull = tmp[tmp[id_col].notna()].copy()
    unique_id = int(nonnull[id_col].nunique())
    dup_cnt = int((nonnull.groupby(id_col).size() > 1).sum())
    return {'source': source, 'rows': rows, 'null_id_cnt': null_id, 'unique_id': unique_id, 'duplicated_id_cnt': dup_cnt}

id_summary = pd.DataFrame([
    id_stats(excel_ids_df, 'excel_agr_id', 'excel'),
    id_stats(ods_ids_sa_active_df, 'lake_agr_id', 'ods_sa_active'),
    id_stats(ods_ids_all_df, 'lake_agr_id', 'ods_all'),
])
display(id_summary)

excel_id_set = set(excel_ids_df['excel_agr_id'].dropna().tolist())
lake_id_set = set(ods_ids_sa_active_df['lake_agr_id'].dropna().tolist())

cross_id_summary = pd.DataFrame([{
    'common_id': len(excel_id_set & lake_id_set),
    'only_excel_id': len(excel_id_set - lake_id_set),
    'only_lake_id': len(lake_id_set - excel_id_set),
}])
display(cross_id_summary)

pick_only_excel_id = sorted(list(excel_id_set - lake_id_set))[:2]
pick_only_lake_id = sorted(list(lake_id_set - excel_id_set))[:2]

print('only_excel_id (2):', pick_only_excel_id)
display(excel_ids_df[excel_ids_df['excel_agr_id'].isin(pick_only_excel_id)].sort_values(['excel_agr_id', 'inn', 'contract_number']).drop_duplicates())
display(ods_ids_sa_active_df[ods_ids_sa_active_df['lake_agr_id'].isin(pick_only_excel_id)].sort_values(['lake_agr_id', 'inn', 'contract_number']).drop_duplicates())

print('only_lake_id (2):', pick_only_lake_id)
display(excel_ids_df[excel_ids_df['excel_agr_id'].isin(pick_only_lake_id)].sort_values(['excel_agr_id', 'inn', 'contract_number']).drop_duplicates())
display(ods_ids_sa_active_df[ods_ids_sa_active_df['lake_agr_id'].isin(pick_only_lake_id)].sort_values(['lake_agr_id', 'inn', 'contract_number']).drop_duplicates())

## H) Excel alternative key check (`inn+contract`, `+name_strict`, `+name_soft`)

In [ ]:
excel_keys_df = excel_df.copy()

# strict: как есть после normalize_text
excel_keys_df['name_norm_strict'] = excel_keys_df['client_name'].fillna('')

# soft: ослабленная нормализация имени
soft = excel_keys_df['client_name'].fillna('').str.upper()
soft = soft.str.replace('"', '', regex=False).str.replace("'", '', regex=False)
soft = soft.str.replace('(', ' ', regex=False).str.replace(')', ' ', regex=False)
soft = soft.str.replace(r'\bООО\b|\bАО\b|\bПАО\b|\bЗАО\b|\bИП\b', ' ', regex=True)
soft = soft.str.replace(r'\s+', ' ', regex=True).str.strip()
excel_keys_df['name_norm_soft'] = soft

excel_keys_df['k1_inn_contract'] = excel_keys_df['inn'].fillna('') + '|' + excel_keys_df['contract_number'].fillna('')
excel_keys_df['k2_inn_contract_name_strict'] = excel_keys_df['k1_inn_contract'] + '|' + excel_keys_df['name_norm_strict'].fillna('')
excel_keys_df['k3_inn_contract_name_soft'] = excel_keys_df['k1_inn_contract'] + '|' + excel_keys_df['name_norm_soft'].fillna('')


def key_metrics(df, key_col):
    z = df.copy()
    rows = len(z)
    unique_keys = z[key_col].nunique()
    dup_keys = int((z.groupby(key_col).size() > 1).sum())
    dup_share = round(100.0 * dup_keys / max(unique_keys, 1), 2)
    conflict_rows = (
        z.groupby(key_col, as_index=False)
         .agg(row_cnt=('inn', 'size'),
              uniq_inn=('inn', 'nunique'),
              uniq_contract=('contract_number', 'nunique'),
              uniq_name=('client_name', 'nunique'))
    )
    conflicts = conflict_rows[conflict_rows['row_cnt'] > 1].copy()
    return {
        'key': key_col,
        'rows': rows,
        'unique_keys': unique_keys,
        'duplicate_keys': dup_keys,
        'duplicate_share_pct': dup_share,
        'conflict_keys_cnt': len(conflicts)
    }, conflicts

k1_m, k1_conf = key_metrics(excel_keys_df, 'k1_inn_contract')
k2_m, k2_conf = key_metrics(excel_keys_df, 'k2_inn_contract_name_strict')
k3_m, k3_conf = key_metrics(excel_keys_df, 'k3_inn_contract_name_soft')

key_summary = pd.DataFrame([k1_m, k2_m, k3_m])
display(key_summary)

print('Top conflicts: k1')
display(k1_conf.sort_values('row_cnt', ascending=False).head(20))
print('Top conflicts: k2')
display(k2_conf.sort_values('row_cnt', ascending=False).head(20))
print('Top conflicts: k3')
display(k3_conf.sort_values('row_cnt', ascending=False).head(20))

## I) Key recommendation

In [ ]:
rec_rows = []

k1_dup = float(key_summary[key_summary['key'] == 'k1_inn_contract']['duplicate_share_pct'].iloc[0])
k3_dup = float(key_summary[key_summary['key'] == 'k3_inn_contract_name_soft']['duplicate_share_pct'].iloc[0])

if k1_dup <= 1.0:
    recommendation = 'Использовать k1 = inn + contract_number как базовый ключ. client_name хранить как атрибут.'
elif k3_dup < k1_dup:
    recommendation = 'Для matching использовать k3 (soft name), но бизнес-PK оставить k1 = inn + contract_number.'
else:
    recommendation = 'Оставить k1 как базовый ключ и включить ручной разбор конфликтов по name-вариантам.'

rec_rows.append({'decision': recommendation})

# Примеры конфликтов по k1 (top 2 ключа)
example_k1 = k1_conf.sort_values('row_cnt', ascending=False).head(2)['k1_inn_contract'].tolist() if len(k1_conf) else []
example_rows = excel_keys_df[excel_keys_df['k1_inn_contract'].isin(example_k1)].copy() if example_k1 else pd.DataFrame()

display(pd.DataFrame(rec_rows))
print('Conflict examples by k1 (top 2 keys):')
display(example_rows[['inn', 'contract_number', 'client_name', 'name_norm_soft', 'k1_inn_contract']].head(100))

## J) Пояснение по нормализации ИНН (10/12, только ведущий 0)

Логика:
- валидные длины ИНН: 10 (UL) и 12 (IP/FL);
- если после очистки получилось 9 или 11 — добавляем ведущий `0` (восстановление потери формата Excel);
- добавляем/убираем нули только слева;
- "лишний ноль в конце" автоматически не исправляем, это уже искажение данных, а не форматирование.

In [ ]:
norm_demo = pd.DataFrame({
    'raw_value': ['100004062', '0100004062', '7708044880', '123456789012', '12345678901', '1234567890', '1234567890123'],
})
norm_demo['normalized'] = norm_demo['raw_value'].apply(normalize_id)
norm_demo['inn_type'] = norm_demo['normalized'].apply(inn_type)
display(norm_demo)